<img src="logo.png" alt="Vegeta" width="240">

# Kestrel — the design study

**The one notebook to open for Kestrel**: the zero-stagger biplane quadrotor tailsitter in its three variants —
**4Q-1** (round struts, X bracing, the rotors as the only effectors), **4Q-2** (elevons, faired posts, no diagonals,
the reflexed section) and the proposed **4Q-3** (4Q-2's airframe with crow-brake elevons and reverse-capable drives) —
from the geometry and the mass budget to the transitions and a final assessment, on **inexpensive methods first**:
the mass table, Boreas's blade-element propulsion, the vortex lattice with Prandtl's biplane correlation, the drag
build-up, NeuralFoil's section polars, beam and truss statics, the reduced-order transition model. Where a
higher-fidelity answer exists on disk — the airframe in RANS with the propellers as actuator disks, the frame in
CalculiX — it is **read**, compared and labelled; it is never computed here.

**The rule this notebook follows (`workflows/README.md`).** An analysis expected to take more than about 15–30
minutes — the whole campaign, not one case — is an explicit **workflow**, launched by the engineer from a shell and
monitored through its `run.log`; a notebook may *require* its output and must say exactly how to produce it, and must
never launch a solver, regenerate a missing analysis or hide expensive work behind an import or a loader. The Kestrel
workflows: `workflows/kestrel/airframe_cfd.py` (Aeromant, rotor disks), `workflows/kestrel/structure_fea.py` (Talos);
the resolved propeller stays in **`38_prop_lab`**, whose exported calibration is read here when it passed its gates.

**How every number is labelled.** *analytic* — closed-form physics; *empirical* — a correlation or surrogate with its
source and range (§ 5's table); *solver* — a Vegeta tool's numerical solution on its mesh or lattice; *validated* — a
converged solver result checked against a reference. An attached-flow formula is never a stall model, an unconverged
CFD case never a validation, and a NOT RUN stays NOT RUN with the analytical estimate visible beside it.

| § | section | method |
|---|---|---|
| 1 | prerequisites: what is on disk, its provenance and compatibility | — |
| 2 | geometry and layout, 3 variants | CAD (Dedalus) |
| 3 | mass budget, CG, inertia | estimate (component data + assumed densities) |
| 4 | propulsion and the hover | BEMT (solver) against momentum theory (analytic) |
| 5 | aerodynamics: lattice, drag build-up, sections, the polar; the CFD sweep read against it; the stall decision | solver + empirical; RANS where on file |
| 6 | trim, stability and control authority | lattice + empirical |
| 7 | performance and energy | solver + empirical |
| 8 | structure: plane frame and modes (analytic); FEA where on file | analytic; CalculiX where on file |
| 9 | transitions both ways, the remedies, the CFD correction | reduced-order model (solver) |
| 10 | which workflows to run next | — |
| 11 | final assessment, runtime, export | — |

## 1. Prerequisites

Workflow results are selected **by explicit run directory** in `RUNS`; a path alone proves nothing — each run's
`manifest.json` carries the design fingerprint it was made for (CAD parameters, mass, CG, propeller calibration,
source hashes) and `status.json` its state, and both are checked against this notebook's own build of the design.
**Required** inputs stop the notebook with the command that produces them; **optional** ones show NOT RUN and the
analytical estimate stays visible, labelled. `KESTREL_SCREENING=1` makes everything optional for a screening pass.

Runs live under `workflows/runs/<book>/<chapter>/` — the **book** is the design (`kestrel`, code K), a **chapter**
one numbered study on it (K1 `airframe-sweep`, K2 `airframe-stall`, K3 `airframe-cruise`, K4 `airframe-transitions`
… K9 `frame`, K10 `joint`) — and `wf` (`workflows/cli.py`) shows the tree, launches a chapter by code and watches
it. The first campaign's results (notebook 39, screening quality, 400 iterations) are imported into K1's folder by
`wf import_legacy`; they are what `RUNS` points at below until a workflow run replaces them (`wf notebook <run>`
prints the line).

In [ ]:
import json, math, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0, "designs")
import kestrel, kestrel_systems as ks, kestrel_flight as kf, kestrel_control as kc, kestrel_transition as kt, kestrel_cfd as kcfd
import kestrel_structure as kst, kestrel_drawings as kd, kestrel_loop as kl, kestrel_report as kr, kestrel2 as k2
import kestrel_campaigns as kcp, kestrel_study as kstu
from vegeta import dedalus

pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
CLOCK = kstu.Stopwatch()
OUT = Path("output/43_kestrel_study"); OUT.mkdir(parents=True, exist_ok=True)
MAIN = kstu.MAIN                                   # the variant the required inputs and the final assessment are about
VARIANTS = kstu.VARIANTS
SCREENING = os.environ.get("KESTREL_SCREENING") == "1"

# ---- the explicit selection of workflow results (edit here; nothing is picked automatically) ----
RUNS = {"airframe_cfd": {"4Q-1": {"sweep": "workflows/runs/kestrel/airframe-sweep/legacy_20261016_4Q-1_39_kestrel_tests"},
                         "4Q-2": {"sweep": "workflows/runs/kestrel/airframe-sweep/legacy_20261016_4Q-2_39_kestrel_tests"},
                         "4Q-3": {"sweep": None}},
        "structure_fea": {"4Q-2": None}}
REQUIRED = set() if SCREENING else {("airframe_cfd", MAIN, "sweep"), ("structure_fea", MAIN)}
print(f"main variant {MAIN} | screening {SCREENING} | required: {sorted(REQUIRED) or 'nothing'}")
FP = {v: kcp.fingerprint(v) for v in VARIANTS}
PREREQ, SUMM = kstu.prerequisites(RUNS, REQUIRED, VARIANTS, fingerprints=FP)        # raises PrerequisiteError with the commands when a required input is unusable
display(PREREQ)
avail = kcp.find_runs("airframe_cfd") + kcp.find_runs("structure_fea")
print(f"runs on disk under workflows/runs/<book>/<chapter>/: {len(avail)}" + ("".join(f"\n  {r['book']:9s} {r['chapter']:22s} {r['variant'] or '':5s} {r['state']:10s} {r['run_id']}" for r in avail[:12]) if avail else "") + "\n(wf runs lists them with their chapter codes)")
CLOCK.lap("1 prerequisites")

## 2. Geometry and layout

Both wings share one leading-edge station (zero stagger is built into the CAD, not a parameter); the pod spans the gap
with the camera in the nose and the payload capsule on the axis; four fixed propellers thrust along the pod. The layout
rules — the discs clear each other and the pod, the propeller plane is ahead of the leading edge, the feet reach aft of
the pod — are checked before anything is computed. The variants differ in their effectors, struts, bracing and section;
the wings, pod and rotors are the same, so the lattice is computed once.

In [ ]:
design = dedalus.load_design("designs/kestrel.py:Kestrel4Q")
SETS = kstu.build_all(VARIANTS)                    # per variant: parameters, layout, mass table, CG, polar, neutral point, hover, cruise table, transition vehicle
S = SETS[MAIN]; P, L = S["p"], S["L"]
for v, s in SETS.items():
    print(f"Kestrel-{v}: layout rules {'all hold' if not s['layout_rules'] else s['layout_rules']}")
diff = {v: {k: s["p"][k] for k in s["p"] if s["p"][k] != SETS["4Q-1"]["p"][k] and k not in kestrel.PER_CALL} for v, s in SETS.items()}
display(pd.DataFrame({f"Kestrel-{v}": d for v, d in diff.items()}).fillna("as 4Q-1").rename_axis("parameters that differ"))
lay = pd.Series({k: v for k, v in L.items() if isinstance(v, (int, float))}, name="layout [m, m²]")
display(lay.round(4).to_frame().T)
g = design.generate(part="aircraft", **{k: P[k] for k in ("elevons", "elevon_chord", "elevon_inner_y", "strut_fairing", "bracing")})
dedalus.plot_views(g, figsize=(13, 9)); plt.suptitle(f"Kestrel-{MAIN}", y=0.99); plt.show()
fig, ax = plt.subplots(2, 1, figsize=(12, 9))
(kd.front_view_4q2 if P["elevons"] else kd.front_view)(P, cg=S["ci"], ax=ax[0]); kd.side_view(P, x_cg=S["ci"]["x_cg"], x_np=S["np"]["x_np"], ax=ax[1]); plt.tight_layout(); plt.show()
CLOCK.lap("2 geometry")

## 3. Mass budget, centre of gravity, inertia

Bought parts from `kestrel_systems.COMPONENTS`, the structure from the layout's lengths and areas with assumed
densities (balsa-composite panels, carbon tubes, printed joints, a foam-and-film strut sleeve), a growth margin. The
battery slides on the pod's floor to put the CG at 10 % of the chord behind the leading edge; § 6 says whether that
margin is right. **Method: estimate** — component masses are catalogue values, structural densities are assumptions
(±15 % on the structure's share).

In [ ]:
display(kstu.mass_groups(SETS).round(3))
mt = pd.DataFrame(S["table"]).set_index("item")[["mass_kg", "x", "y", "z", "group"]]
display(mt.assign(mass_g=lambda d: d.mass_kg * 1000).drop(columns="mass_kg").round(3))
CI = S["ci"]
print(f"Kestrel-{MAIN}: take-off mass {CI['mass_kg']:.3f} kg; CG x {CI['x_cg'] * 1000:.1f} mm (battery at {CI['battery_x'] * 1000:.0f} mm, "
      f"{'at its stop' if CI['battery_clipped'] else 'inside its bay'}), z {CI['z_cg'] * 1000:+.1f} mm; Ixx {CI['Ixx']:.4f}, Iyy {CI['Iyy']:.4f}, Izz {CI['Izz']:.4f} kg·m²")
CLOCK.lap("3 mass")

## 4. Propulsion and the hover

One drive: a 9×6 propeller on a 2216-class 880 KV motor on 4S. **Boreas** (blade-element momentum theory with a
generic thin cambered section, the motor's first-order model, the battery) gives the drive tables the transition
model reads: the full thrust $T_{max}(V)$, the torque-to-thrust ratio $\kappa = Q/T$ (the only lever about the thrust
axis without surfaces) and the electrical power $P(T, V)$. The tables are **on file** (`data/kestrel_drive.json`,
keyed on the components and the calibration) — the prerequisite table said whether they would be computed here.

The analytic check is momentum theory: a rotor of disc area $A$ carrying thrust $T$ induces
$v_i = \sqrt{T / 2\rho A}$ and needs the ideal power $P_i = T\,v_i = T^{3/2}/\sqrt{2\rho A}$; the real rotor takes
$P_i / FM$ with a figure of merit of 0.5–0.65 for a 9-inch propeller, and the drive $\approx 0.8$ of that again in
electrical terms. When **prop_lab** (notebook 38) has a calibration on file that passed its gates, the blade section's
$c_l$ and $c_d$ scales and the normal-force factor are applied in `kestrel_systems.propulsion`; without it the generic
section reads ~25 % low on static thrust, which keeps the control margins conservative.

In [ ]:
PL = ks.prop_lab_calibration()
print("prop_lab:", (f"calibration on file — cl ×{PL['cl_scale']:.3f}, cd ×{PL['cd_scale']:.3f}, CNα ×{PL['cn_factor']:.3f}" if PL["on_file"] else "NOT RUN or rejected: the generic blade section"))
DR = S["veh"].drive
display(pd.DataFrame({f"Kestrel-{v}": s["hover"] for v, s in SETS.items()}).T.round(4).rename_axis("hover, one drive [Boreas BEMT, solver]"))
display(pd.Series(kstu.hover_analytic(S), name=f"Kestrel-{MAIN}: momentum theory [analytic] beside Boreas").round(3).to_frame())
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(DR.V, DR.T_max, "o-"); ax[0].axhline(CI["mass_kg"] * ks.G / 4, color="k", ls=":", label="¼ of the weight"); ax[0].set(xlabel="axial inflow [m/s]", ylabel="N", title="full thrust, one rotor (BEMT)"); ax[0].legend()
ax[1].plot(DR.V, np.array(DR.kappa) * 1000, "o-"); ax[1].set(xlabel="axial inflow [m/s]", ylabel="mm", title="κ = Q / T")
for v in DR.V[::3]:
    j = list(DR.V).index(v); ax[2].plot(DR.T, DR.P[j], label=f"{v:g} m/s")
ax[2].set(xlabel="thrust [N]", ylabel="W", title="electrical power, one drive", ylim=(0, 400)); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()
CLOCK.lap("4 propulsion")

## 5. Aerodynamics: the lattice, the drag build-up, the sections, the polar — and the CFD read against them

**Lift.** PEREGRINE's vortex lattice on the two wings at $z = \pm \text{gap}/2$ gives the pair's lift slope and span
efficiency; Prandtl's biplane theory puts the induced drag at $(1+\sigma)/2$ of the monoplane's with $\sigma$ from
gap/span — the lattice should land near it. **Drag** is a build-up: skin friction with form factors on the wetted
areas, the struts as cylinders (or faired, 0.12 on the tube's frontal area), an interference allowance of 10 %,
$C_{D} = C_{D0} + k\,C_L^2$ with $k = 1/(\pi\,AR\,e)$ on the pair's effective aspect ratio. **The section** sets what
the lattice cannot: $c_{l,max}$ (NeuralFoil at Re $1.5\cdot10^5$, $\times 0.9$ for a built-up wing), $\alpha_0$, the
section drag and the elevon's flap effectiveness $\tau$. Beyond the stall the polar is blended into a flat plate
(Viterna) — a time-averaged post-stall load model, **not** a model of stall onset.

The table of correlations states each one's source, applicability and uncertainty.

In [ ]:
display(kstu.correlations_table())
A = S["aero"]
display(pd.Series(kstu.biplane_check(A, L), name="biplane: Prandtl [empirical] against the lattice [solver]").round(4).to_frame().T)
D1, D2 = SETS["4Q-1"]["drag"], SETS[MAIN]["drag"]
dtab = pd.DataFrame({"4Q-1 round struts, X": D1["parts_m2"], f"{MAIN} faired posts": D2["parts_m2"]}) * 1e4
display(dtab.round(2).rename(columns=lambda c: c + " [cm²]").rename_axis("drag areas [empirical build-up]"))
print(f"Cd0: 4Q-1 {D1['Cd0']:.4f} (bracing {D1['bracing_share'] * 100:.0f} %), {MAIN} {D2['Cd0']:.4f} (bracing {D2['bracing_share'] * 100:.0f} %)")
display(pd.DataFrame({f"Kestrel-{v}": s["pol"].__dict__ for v, s in SETS.items()}).T.rename_axis("the polars [lattice + build-up + NeuralFoil]"))

**The CFD sweep read against the polar.** For each variant with an airframe run on file, the bare airframe's RANS
coefficients (0–90° at 10 m/s, k-ω SST, the propellers off) against the model's at the same angles: lift slope,
$C_{D}$ at the lowest angle, the static margin from $C_{m\alpha}$, and the largest $C_L$ below 20° on the sweep's
angles — a lower bound on CL max on a 5° grid. The label says what the run is: **validated** only when every bare
case reached its residual target; the first campaign (400 iterations) did not, so its numbers are *indicative*. The
stall table turns CL max into the stall speed, the cruise margin and the wing area that would restore a 1.3 × margin —
the decision the fine `stall` campaign is for.

In [ ]:
CFD = {v: kstu.cfd_sweep(SUMM["airframe_cfd"].get(v, {}).get("sweep"), SETS[v]) for v in VARIANTS}
for v, c in CFD.items():
    print(f"Kestrel-{v}: CFD sweep {c['label']}" + (f" — {c['n_converged']} of {c['n_bare']} bare cases converged; {c.get('quality')} quality; {c.get('provenance', {}).get('notebook') or 'workflow run'}" if c["n_bare"] else ""))
    if c["cmp"] is not None and len(c["cmp"]):
        display(c["cmp"].drop(columns=["note"]).astype({"α [deg]": float}).round(3).rename_axis(f"Kestrel-{v}: CFD [{c['label']}] against the model"))
        display(pd.DataFrame(c["numbers"]).round(3))
c = CFD[MAIN]
kd.polar_figure(S["pol"], c["rt"] if c["rt"] is not None and len(c["rt"]) else None); plt.title(f"Kestrel-{MAIN}: the model's polar" + (f" and the CFD sweep ({c['label']})" if c["n_bare"] else "")); plt.show()
STALL = kstu.stall_decision(SETS, CFD)
display(STALL.T)
CLOCK.lap("5 aerodynamics")

## 6. Trim, stability and control authority

No tail: the neutral point is the wing pair's aerodynamic centre moved by the pod (Raymer's fuselage term) and by the
four propellers ahead of the CG (De Young's normal-force slope, destabilising); the static margin is what the CG
leaves. Level flight needs a nose-up moment at positive lift: thrust alone trims by the lower pair pushing harder
than the upper, which fails once the drag is small (the upper pair would have to drop below idle) — then the elevons
trim. The hold point in the hover: the allocation matrix has full rank only with the diagonal pairs counter-rotating;
the heading lever is the reaction torque $\kappa T$ alone, the elevons in the slipstream multiply it.

In [ ]:
display(pd.DataFrame({f"Kestrel-{v}": s["np"] for v, s in SETS.items()}).T.round(4).rename_axis("neutral point [lattice + empirical]"))
display(kstu.trim_table(S).round(3).rename_axis(f"Kestrel-{MAIN}: level flight and the trim [solver + empirical]"))
CT = kstu.control_table(S)
display(CT["rank"]); display(CT["hover authority"].round(3))
if "elevon authority" in CT:
    display(pd.Series(CT["elevon hinge"], name="elevon hinge moments").round(4).to_frame().T); display(CT["elevon authority"].round(3))
CLOCK.lap("6 trim")

## 7. Performance and energy

Level flight against airspeed: drag, the rotors' full thrust falling with the inflow and where it meets the drag
(the thrust-limited maximum speed — the 9×6's pitch speed, not the drag, sets it), the electrical power, endurance and
range on 80 % of the battery; specific excess power $P_s = (T - D)\,V/W$ for the climb; the turn at the corner speed
where the lift limit meets the 4 g structural limit; the horizontal-plane authority. The energy budget charges the
hover for take-off, landing and the transitions before the cruise. **Method: solver (BEMT, lattice) + empirical
(build-up, ±20 % on Cd0 → ±15 % on endurance).**

In [ ]:
PERF = kstu.performance(SETS)
display(PERF["pursuit"].astype(float).round(2).rename_axis("pursuit table"))
fig, axs = plt.subplots(1, 3, figsize=(17, 4.6))
kd.drag_vs_speed({f"Kestrel-{v}": s["veh"] for v, s in SETS.items()}, speeds=np.arange(8.0, 40.5, 0.5), axs=axs); plt.tight_layout(); plt.show()
fig, axs = plt.subplots(1, 2, figsize=(14, 4.2))
for n, pf in PERF["perfs"].items():
    t = pf["table"]; axs[0].plot(t.index, t["endurance [min]"], label=n); axs[1].plot(t.index, t["P_s (climb rate) [m/s]"], label=n)
axs[0].set(xlabel="V [m/s]", ylabel="min", title="endurance in level flight (80 % of the battery)"); axs[1].set(xlabel="V [m/s]", ylabel="m/s", title="specific excess power: climb rate"); axs[1].axhline(0, color="k", lw=0.6)
for a_ in axs: a_.legend(fontsize=8)
plt.tight_layout(); plt.show()
display(pd.Series(kstu.energy_budget(S), name=f"Kestrel-{MAIN}: energy budget").round(2).to_frame())
CLOCK.lap("7 performance")

## 8. Structure

**Screening (analytic).** The front view as a plane frame at a 4 g pull-up: the spars as beams clamped at the pod,
the posts and diagonals as pin-ended bars, the lift elliptic less the wings' and motors' inertia — even, and with the
upper wing carrying 75 % (one wing stalling first); the spar stress $\sigma = M c / I$ and its reserve against the
tube's allowable, Euler buckling of the struts, the propeller gap's change; the motor stations' stiffness and first
modes per bracing against the pitch loop's bandwidth and the rotor's 1P; the aircraft standing on its feet in wind.

**FEA (solver), where on file.** The same frame as solid tubes in CalculiX under the same loads at ultimate (× 1.5) —
A the pull-up, B the 75 % wing, C the heading manoeuvre's torsion, the modes — and the printed spar–post joint with
its across-layer tension; read against the plane frame's numbers. Produced by `workflows/kestrel/structure_fea.py`;
NOT RUN here means exactly that.

In [ ]:
ST = kstu.structure_screen(S)
display(ST["even lift"].round(3).rename_axis("plane frame, 4 g, even lift [analytic]")); display(ST["upper wing 75 %"].round(3).rename_axis("plane frame, 4 g, upper wing 75 % [analytic]"))
display(ST["modes"].round(2)); print("references:", {k: round(v, 2) for k, v in ST["modes"].attrs["references"].items()})
display(pd.Series({k: v for k, v in ST["stance"].items() if k != "edges"}, name="standing on the feet [analytic]").round(2).to_frame().T)
FEA_SUM = SUMM["structure_fea"].get(MAIN)
FEAT = kstu.fea_table(FEA_SUM)
if FEAT is None:
    print(f"FEA of Kestrel-{MAIN}: NOT RUN — {kcp.command('structure_fea', variant=MAIN, campaign='all')}\n  the plane frame's reserves above stand in (analytic); nothing below is a strength verdict")
    display(ST["reference"].astype({c: float for c in ST["reference"].columns if c not in ("bracing", "case")}).round(2).rename_axis("plane frame reference the FEA will be read against"))
else:
    print(f"FEA of Kestrel-{MAIN} [solver]: {FEA_SUM['counts']}")
    display(FEAT)
    cmpf = kstu.fea_against_frame(FEA_SUM, S)
    if cmpf is not None:
        display(cmpf.astype(float).round(2).rename_axis("FEA against the plane frame"))
CLOCK.lap("8 structure")

## 9. The transitions — hover to cruise and back, through the stall

`kestrel_transition` flies the aircraft in the pitch plane: each wing split into the span blown by its propellers
(Glauert's slipstream adds $\Delta v$ along the body axis) and the free-stream span, the Viterna polar beyond the
stall, a pitch PID scheduled on airspeed, the height held by the collective, the moment allocated to differential
thrust and the elevons, roll checked against one side's free-stream span stalling first. **Method: reduced-order
model (solver)** — its post-stall aerodynamics are the Viterna blend, so its verdicts are comparisons between
variants and remedies, not certified trajectories; the blown stalled regime is what the CFD `transitions` campaign is
for. The criteria: height within 3 m below and 10 m above the start on the way back, pitch within 10°, roll demand
within the authority, no divergence.

Only the decision-relevant episodes are flown (each is ~8 s): every variant forward and back at the 3 s schedule in
still air and back into a 6 m/s headwind; 4Q-2 and 4Q-3 back with the remedies notebook 42 found (reverse thrust on the
lower pair, the crow brake with it); the main variant again on the CFD's correction when a sweep is on file.

In [ ]:
CORR = kcp.airframe_correction(SUMM["airframe_cfd"].get(MAIN, {}).get("sweep") or {}, S["veh"], source="sweep") if SUMM["airframe_cfd"].get(MAIN) else None
if CORR is not None:
    print(f"CFD correction for Kestrel-{MAIN}: from the {CORR.source} ({len(CORR.samples)} points, {CFD[MAIN]['label']})")
TR = kstu.transitions(SETS, main=MAIN, cfd_correction=CORR)
TRT = kstu.transition_columns(TR["table"])
display(TRT.round(2))
for kind in ("forward", "back"):
    kd.episode_figure(TR["episodes"][(MAIN, kind, "as is, still air")], S["veh"], title=f"Kestrel-{MAIN}, {kind}, 3 s, still air"); plt.savefig(OUT / f"{MAIN}_{kind}_3s.png", dpi=110, bbox_inches="tight"); plt.show()
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, kind in zip(axs, ("forward", "back")):
    for v in VARIANTS:
        ep = TR["episodes"][(v, kind, "as is, still air")]
        ax.plot(ep.log["X"], ep.log["Z"], label=f"Kestrel-{v}: gain {ep.metrics['height gain [m]']:.1f} m, loss {ep.metrics['height loss [m]']:.1f} m, pass {ep.metrics['pass']}")
    for key in [k for k in TR["episodes"] if k[0] == MAIN and k[1] == kind and not k[2].startswith("as is")]:
        ep = TR["episodes"][key]; ax.plot(ep.log["X"], ep.log["Z"], "--", label=f"Kestrel-{MAIN}, {key[2]}: gain {ep.metrics['height gain [m]']:.1f} m, pass {ep.metrics['pass']}")
    ax.axhline(30, color="k", lw=0.6, ls=":"); ax.set(ylabel="height [m]", title=f"{kind} transition, 3 s schedule, still air"); ax.legend(fontsize=8)
axs[1].set_xlabel("x [m]"); plt.tight_layout(); plt.savefig(OUT / "paths.png", dpi=110); plt.show()
DEC = kstu.deceleration(SETS)
display(DEC.round(2).rename_axis(["vehicle", f"level deceleration {kf.V_CRUISE:g} → 13 m/s"]))
CLOCK.lap("9 transitions")

## 10. Which workflows to run next

The screening above says which solver cases change a decision. They are listed with their reasons and commands; an
engineer launches them from a shell (detached: `nohup … &` or `tmux`, then `tail -F <run>/run.log`), and points `RUNS`
at the run directory each prints. Compatible results are reused, never recomputed: a run whose fingerprint matches the
design stays valid until the CAD parameters, the mass or the propeller calibration change.

In [ ]:
REC = kstu.recommended_campaigns(SETS, CFD, SUMM["structure_fea"], main=MAIN)
display(REC.drop(columns=["command"]))
print("\n".join(f"{i}. {r['command']}" for i, r in REC.iterrows()))
CLOCK.lap("10 recommendations")

## 11. Final assessment

The checks of the main variant against the study's targets, each with the method behind it; the verdict lines are
written from the numbers above. A screening pass (`KESTREL_SCREENING=1`) is labelled unvalidated. The runtime of this
notebook is measured, not claimed, and exported with the report.

In [ ]:
ASSESS = kstu.assessment(SETS, CFD, SUMM["structure_fea"], TR, PERF, main=MAIN, screening=SCREENING)
display(ASSESS["table"])
print("\n".join(ASSESS["lines"]))
display(kstu.variants_table(SETS).T)
CLOCK.lap("11 assessment")
RT = CLOCK.frame(); display(RT.T)
print(f"measured runtime: {CLOCK.total():.0f} s on {os.uname().nodename} ({os.cpu_count()} CPUs) — the workflow threshold is {kstu.WORKFLOW_THRESHOLD_MIN[0]}–{kstu.WORKFLOW_THRESHOLD_MIN[1]} min")

### Export

The numbers as JSON (`designs/data/reports/43_kestrel_study.json`) and Markdown, stitched into the dossier
`designs/data/kestrel_dossier.json` / `.md` — paste the JSON into a conversation and the design continues from it.

In [ ]:
REPORT = {"prerequisites": PREREQ, "parameters that differ": pd.DataFrame({f"Kestrel-{v}": d for v, d in diff.items()}), "layout [m, m²]": lay, "mass groups [kg]": kstu.mass_groups(SETS),
          "mass, CG, inertia (main)": {k: v for k, v in CI.items() if k != "I"}, "hover (Boreas)": pd.DataFrame({f"Kestrel-{v}": s["hover"] for v, s in SETS.items()}).T,
          "hover (momentum theory)": kstu.hover_analytic(S), "correlations": kstu.correlations_table(), "biplane check": kstu.biplane_check(A, L), "drag areas [cm²]": dtab,
          "polars": pd.DataFrame({f"Kestrel-{v}": s["pol"].__dict__ for v, s in SETS.items()}).T,
          "CFD sweep against the model": {v: c["cmp"] for v, c in CFD.items() if c["cmp"] is not None}, "CFD sweep numbers": {v: c["numbers"] for v, c in CFD.items() if c["numbers"]},
          "CFD labels": {v: {"label": c["label"], "bare cases": c["n_bare"], "converged": c["n_converged"]} for v, c in CFD.items()}, "stall decision": STALL,
          "neutral point": pd.DataFrame({f"Kestrel-{v}": s["np"] for v, s in SETS.items()}).T, "trim (main)": kstu.trim_table(S), "allocation rank": CT["rank"], "hover authority": CT["hover authority"],
          "pursuit": PERF["pursuit"], "energy budget (main)": kstu.energy_budget(S), "plane frame, even lift": ST["even lift"], "plane frame, 75 % wing": ST["upper wing 75 %"], "stiffness and modes": ST["modes"],
          "ground stance": {k: v for k, v in ST["stance"].items() if k != "edges"}, "FEA": FEAT if FEAT is not None else "NOT RUN", "transitions": TRT, "level deceleration": DEC,
          "recommended workflows": REC, "final checks": ASSESS["table"], "verdict": ASSESS["lines"], "variants": kstu.variants_table(SETS), "runtime [s]": RT, "status": {"screening": SCREENING, "required": sorted(map(str, REQUIRED))}}
EXP = kr.export(kstu.NAME, "Kestrel — the design study (notebook 43)", REPORT, meta={"main": MAIN, "mass [kg]": CI["mass_kg"], "runtime_s": CLOCK.total(), "screening": SCREENING,
                                                                                    "CFD": {v: c["label"] for v, c in CFD.items()}, "FEA": "on file" if FEAT is not None else "NOT RUN"})
print("JSON report:", EXP["json"], f"({EXP['json_kB']} kB) | dossier:", EXP["dossier_json"], f"({EXP['dossier_json_kB']} kB) | Markdown:", EXP["markdown"])